### Define parameters

# User Configuration

In [ ]:
# Set these paths before running the notebook

results_indir = ""  
rsa_model_rdm_dir = ""  
fmri_dir = ""  
hands_annotations_dir = ""  
cvat_annotations_dir = "" 

In [ ]:
subjects = ['sub01', 'sub02','sub03','sub04','sub05','sub06']


### Define viz parameters

In [ ]:
clean_modelnames_dictmap = {
    'semantic_BGE' : 'Semantic',
    'visual' : 'Visual',
    'roleagnostic_affordance' : 'Action Affordances',
    'hands_linearcomb_affordance': 'Hand Posture Affordances'
    }

abbrev_modelnames_dictmap = {
    'semantic_BGE' : 'Sem',
    'visual' : 'Vis',
    'roleagnostic_affordance' : 'ActAff',
    'hands_linearcomb_affordance': 'HandsAff'
    }
    
clean_exemplarnames_dictmap = {'pan': 'pan',
 'knife': 'knife',
 'cup/mug': 'mug',
 'pot': 'pot',
 'fork': 'fork',
 'spatula': 'spatula',
 'pizza': 'pizza',
 'fridge': 'fridge',
 'drawer': 'drawer',
 'onion': 'onion',
 'cupboard': 'cupboard',
 'package/wrapper': 'wrapper',
 'scissors': 'scissors',
 'white-sauce': 'sauce',
 'pizza-box': 'pizza box',
 'microwave': 'microwave',
 'spoon': 'spoon',
 'plate': 'plate',
 'knob': 'knob',
 'pasta': 'pasta',
 'sink': 'sink',
 'oven/stove': 'stove',
 'oven-mitts': 'oven mitts',
 'carton/canister': 'carton/canister',
 'counter': 'counter',
 'colander': 'colander',
 'sponge/rag':'sponge/rag'}


import matplotlib as mpl
mpl.rcParams['font.family'] = 'Arial'


In [ ]:
import os, sys

# Resolve utility paths relative to this notebook's directory
_nb_dir = os.path.abspath('')
sys.path.append(os.path.join(_nb_dir, 'utils', 'fmri'))
sys.path.append(os.path.join(_nb_dir, 'utils', 'fmri', 'speechmodeltutorial'))
sys.path.append(os.path.join(_nb_dir, 'utils', 'fmri', 'utils_natcook'))

import os, glob, itertools, sys, pickle, ast
from os.path import join
from tqdm import tqdm

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from nilearn import image
from nilearn.masking import apply_mask, unmask
from nilearn.image import math_img


from scipy.stats import spearmanr
from scipy.spatial import distance
from scipy.spatial.distance import squareform


# ======================================= Custom imports =============================================

# -- Local fmri-general functions
from banded_ridge_reconstruct_fir_weights import banded_ridge_reconstruct_fir_weights

# -- Natcook-specific helpers
from FMRIPathConfig import FMRIPathConfig

# ======================================= Input directories =============================================

# --- Define directories and paths
path_config = FMRIPathConfig(fmri_dir)
display('Path templates:', path_config.patterns)

# ======================================= Fixed parameters =============================================
# --- Other
n_jobs = -1
np.random.seed(42)


### Create the model RDMs

Helper functions


In [ ]:
def read_embs_and_make_rdms(embs_path, exemplars):
    with open(embs_path, 'rb') as f:
        embs_dict = pickle.load(f)
    
    # Check which exemplars are actually present
    available_exemplars = [ee for ee in exemplars if ee in embs_dict]
    missing_exemplars = [ee for ee in exemplars if ee not in embs_dict]
    
    if missing_exemplars:
        print(f'\n**WARNING**: {len(missing_exemplars)} exemplars missing from {os.path.basename(embs_path)}: {missing_exemplars}')
        print(f'    Using only {len(available_exemplars)} available exemplars for this model')
    
    # Create the RDM using only available exemplars
    embs_arr = np.array([embs_dict[ee] for ee in available_exemplars])
    rdm = distance.pdist(embs_arr, metric='correlation')
    print(f'{os.path.basename(embs_path)} RDM squared has shape {distance.squareform(rdm).shape}')
    
    return rdm, embs_arr, embs_dict, available_exemplars


def subset_rdm(rdm_condensed, indices):
    """
    Subset a condensed RDM to only include specified exemplar indices.
    
    Parameters
    ----------
    rdm_condensed : array
        Condensed distance matrix from pdist
    indices : list of int
        Indices of exemplars to keep (in desired order)
    
    Returns
    -------
    rdm_subset : array
        Condensed RDM containing only the specified exemplars
    """
    rdm_square = distance.squareform(rdm_condensed)
    rdm_square_subset = rdm_square[np.ix_(indices, indices)]
    return distance.squareform(rdm_square_subset)

In [ ]:

# -------------------- Read exemplars list ( overlap in targets and objects)  -------------------- 
feats_legend_path = join(os.path.dirname(results_indir), 'feats_legend.p')
with open(feats_legend_path, 'rb') as f:
    feats_legend = pickle.load(f)


# define exemplars as the list of targets
exemplars = [i.split('_')[-1] for i in feats_legend if i.startswith('target_')]
exemplars = [i for i in exemplars if 'hand' not in i]

print('Num exemplars = ', len(exemplars))
print(exemplars)
print('')


# -------------------------------------- Read the RDMs --------------------------------------------
# Initialize dictionaries to store both RDMs and their exemplars
all_rdms = {}
all_rdm_exemplars = {}  # Track which exemplars each model has


# ********** Visual **********
visual_embs_path = join(rsa_model_rdm_dir, 'visual/visual_embs.p')
visual_rdm, visual_embs_arr, visual_embs_dict, visual_available_exemplars = read_embs_and_make_rdms(visual_embs_path, exemplars)

all_rdms['visual'] = visual_rdm
all_rdm_exemplars['visual'] = visual_available_exemplars


# ********** semantic BGE **********
# Read embs
semantic_BGE_embs_path = join(rsa_model_rdm_dir, 'semantic/semantic_BGE_embs.p')
with open(semantic_BGE_embs_path, 'rb') as f:
    semantic_embs_dict_editedNames = pickle.load(f)

# Exception for semantic, we need to rename the embs since we had used alternate words to find the semantic embs
df_exemplar_names = pd.read_excel(join(rsa_model_rdm_dir, 'semantic/exemplar_names.xlsx'))
dict_rename_map = {row['exemplars_renamed'] : row['exemplars_in_targets_and_objects'] for idx,row in df_exemplar_names.iterrows()}

semantic_embs_dict = {}
for old_key,val in semantic_embs_dict_editedNames.items():
    new_key = dict_rename_map[old_key]
    semantic_embs_dict[new_key] = val

# Create the RDM
semantic_embs_arr = np.array([semantic_embs_dict[ee] for ee in exemplars])
semantic_rdm = distance.pdist(semantic_embs_arr, metric='correlation')
print('semantic RDM squared has shape ', distance.squareform(semantic_rdm).shape)

all_rdms['semantic_BGE'] = semantic_rdm
all_rdm_exemplars['semantic_BGE'] = exemplars  # semantic has all exemplars



# ********** ROLE-AGNOSTIC Affordances **********
roleagnostic_affordance_embs_path = join(rsa_model_rdm_dir, 'affordance/roleagnostic_affordance_embs.p')
roleagnostic_affordance_rdm, roleagnostic_affordance_embs_arr, roleagnostic_affordance_embs_dict, roleagnostic_available_exemplars = read_embs_and_make_rdms(roleagnostic_affordance_embs_path, exemplars)

all_rdms['roleagnostic_affordance'] = roleagnostic_affordance_rdm
all_rdm_exemplars['roleagnostic_affordance'] = roleagnostic_available_exemplars




# ********** Hand linearcomb Affordances **********
hands_affordance_linearcomb_embs_path = join(rsa_model_rdm_dir, 'hands_affordance/hands_affordance_linearcomb_embs.p')
hands_affordance_linearcomb_rdm, hands_affordance_linearcomb_embs_arr, hands_affordance_linearcomb_embs_dict, hands_linearcomb_available_exemplars = read_embs_and_make_rdms(hands_affordance_linearcomb_embs_path, exemplars)

all_rdms['hands_linearcomb_affordance'] = hands_affordance_linearcomb_rdm
all_rdm_exemplars['hands_linearcomb_affordance'] = hands_linearcomb_available_exemplars




## Load fMRI embeddings:

In [ ]:
from nilearn.image import new_img_like

outpath = f'fmri_embs_TopVox.p'

if os.path.exists(outpath):
    
    print(f'{outpath} found, reading from file.')
    # Save fmri_embs to file to avoid recomputing on subsequent runs
    with open(outpath,'rb') as f:
       fmri_embs = pickle.load(f)

else:

    fmri_embs = {'objects':[], 'targets':[]}
    for subj in subjects:

        print(f'\n\n     {subj}')
        # ------------------- Read subject's encoding model results ---------------------- 
        ridge_results_path =  join(results_indir, subj,  f'{subj}_ridge_results.p')
        # read banded ridge results
        with open(ridge_results_path, 'rb') as f:
            banded_ridge_results = pickle.load(f)

        # Get the anat image and the brainmask
        brainmask_img = image.load_img(path_config.get_brainmask_path(subj))
        anat_img = image.load_img(path_config.get_anat_path(subj))

        # Get the mask of modelled voxels and significant voxels
        mask_modelled_voxels = banded_ridge_results['mask_modelled_voxels'] # shape (237332,)
        sig_mask_fdr = banded_ridge_results['fdr_results']['sig_mask_fdr'] # shape (128436,)

        # Create the 3d mask of significant voxels
        mask_modelled_voxels_img = unmask(mask_modelled_voxels, brainmask_img) # reconstruct the 3d mask of the voxels we modeled
        sig_mask_fdr_img = unmask(banded_ridge_results['fdr_results']['sig_mask_fdr'], 
                        mask_modelled_voxels_img)  # reconstruct 3d mask of significant voxels (a subset of the modeleld voxels, obviously)

        # --------------- Reconstruct the weights (scale and un-delay) ------------------------
        coefs = banded_ridge_reconstruct_fir_weights(banded_ridge_results, verbose=False)


        # ------------------- Compute the winner-takes-all map -----------------------------
        # indices in the map will correspond to indice in feature_space_names
        feature_space_names = banded_ridge_results['feature_space_names']

        # get the split_scores, and then get a categorical map of which was maximally explaining each voxel
        split_scores = banded_ridge_results['split_scores']
        wta = np.argmax(split_scores, axis=0)  # shape (n_voxels,) a label per voxel (e.g., 0 = tools, 1 = objects, …).

        # reconstruc the result to 3d ni space
        wta_img = unmask(wta, brainmask_img)

        # mask out non significant voxels
        def replace_ns_with_nan(img, mask):
            data = img.get_fdata().copy()
            data[~mask.get_fdata().astype(bool)] = np.nan  # set non-significant voxels to NaN
            out = new_img_like(img, data)
            return out

        wta_img_sig = replace_ns_with_nan(wta_img, sig_mask_fdr_img) # replace with nan instead of 0s since 0 is meaningful here
        # now wta_img_sig is an image with numbers corresponding to the index of feature_space_names that won. NaN are non sig voxels

        # convert ceofs to an img for easier masking below
        # no need to mask by significant voxels yet, since the wta map that we will use as a mask already restricts non-sigs
        coefs_img = unmask(coefs, mask_modelled_voxels_img) # this has now shape (96, 114, 96, 173) -> one 3D img per feature (173 features)

        # For each feature group (targets, objects) subset the coeffs by the winning voxels
        def mask_coeffs_by_featurespace(feature_space_name, feature_space_names=feature_space_names, wta_img_sig=wta_img_sig, coefs_img=coefs_img) :
            # Get the index in the list feature_space_name for current featspace: this will correspond to the voxel values where this space won
            featurespace_index = feature_space_names.index(feature_space_name)

            # Create a mask where True are the voxels corresponding to winning voxels of the current feature space
            mask_featspace_wta_img = math_img(f'wta_img_sig == {featurespace_index}', wta_img_sig=wta_img_sig) # True for all voxels that won for currrent featspace
            
            # mask out the coefficient map by the selected voxels 
            coefs_at_wta = apply_mask(coefs_img, mask_featspace_wta_img)
            
            return coefs_at_wta # this has shape n_coefs, n_winning_voxels_for_current_feat

        coefs_targets = mask_coeffs_by_featurespace('target_features')
        coefs_objects = mask_coeffs_by_featurespace('object_features')

        print(f'{subj}: Num of best voxels for TARGETS = {coefs_targets.shape[1]}')
        print(f'{subj}: Num of best voxels for OBJECTS = {coefs_objects.shape[1]}')


        # ------------------ Get coefficients for the features we want to compare ----------------
        # subj_feats_legend used in case feats_legends differs across subjects
        subj_feats_legend = banded_ridge_results['feats_legend']

        # get indices for each feature group
        idx_targets = [subj_feats_legend.index(f'target_{i}') for i in exemplars] 
        idx_objects = [subj_feats_legend.index(f'object_{i}') for i in exemplars]

        # get the coefs of each feature group
        coefs_targets = coefs_targets[idx_targets,:]
        coefs_objects = coefs_objects[idx_objects,:]
            
        print('')
        print('coefs_targets.shape = ', coefs_targets.shape)
        print('coefs_objects.shape = ', coefs_objects.shape)

        # ---------------------- append to group dict ----------------------------------
        fmri_embs['targets'].append(coefs_targets)
        fmri_embs['objects'].append(coefs_objects)

    # Save fmri_embs to file to avoid recomputing on subsequent runs
    with open(outpath,'wb') as f:
        pickle.dump(fmri_embs, f)

print('Done.')

#### Make the RDMs from the loaded fmri embs

In [ ]:
fmri_rdms = {} # for each emb_type there will be a list of rdms, one per subject

for emb_type in fmri_embs.keys():
    print(emb_type)
    fmri_rdms[emb_type] = [] # initialize list for each embedding type
    for ii_subject, subj in enumerate(subjects):
        
        coefs = fmri_embs[emb_type][ii_subject]  # shape (n_exemplars, n_sig_voxels)
        
        # ----------------------- Create RDMs ----------------------------------
        rdm = distance.pdist(coefs, metric='correlation')
        
        fmri_rdms[emb_type].append(rdm)
        print('    rdm shape: ', rdm.shape)
        
    # convert rdms to np array
    fmri_rdms[emb_type] = np.array(fmri_rdms[emb_type])

# Sanity check: objects and targets RDMs should be different
assert ~np.array_equal(fmri_rdms['objects'], fmri_rdms['targets']), 'Objects and targets RDMs are identical!'

## RSA analysis

In [ ]:
# for plotting significance
PVAL_TO_STARS = {
    0.0001: '***',
    0.001: '**',
    0.05: '*'
}


n_subj = fmri_rdms['targets'].shape[0]
model_names = all_rdms.keys()

# ------------------------------------------------------------
# Compute subject-wise correlations
# ------------------------------------------------------------
corrs_targets = np.zeros((n_subj, len(model_names)))
corrs_objects = np.zeros((n_subj, len(model_names)))

for idx_subj in range(n_subj):
    rdm_target = fmri_rdms['targets'][idx_subj]
    rdm_object = fmri_rdms['objects'][idx_subj]

    for i, modelname in enumerate(model_names):
        model_rdm = all_rdms[modelname]
        model_exemplars = all_rdm_exemplars[modelname]
        
        # Subset fMRI RDM if model has missing exemplars
        if len(model_exemplars) < len(exemplars):
            idx_available = [exemplars.index(ee) for ee in model_exemplars]
            rdm_target_subset = subset_rdm(rdm_target, idx_available)
            rdm_object_subset = subset_rdm(rdm_object, idx_available)
            
            if idx_subj == 0:  # Only print for first subject
                print(f"\n{modelname}: Using {len(model_exemplars)}/{len(exemplars)} exemplars")
                print(f"    Missing: {[ee for ee in exemplars if ee not in model_exemplars]}")
        else:
            rdm_target_subset = rdm_target
            rdm_object_subset = rdm_object
        
        rho_targets, _ = spearmanr(model_rdm, rdm_target_subset)
        rho_objects, _ = spearmanr(model_rdm, rdm_object_subset)
        corrs_targets[idx_subj, i] = rho_targets
        corrs_objects[idx_subj, i] = rho_objects

# ------------------------------------------------------------
# Prepare data for plotting (used by both analyses)
# ------------------------------------------------------------
plot_data = []
for i, modelname in enumerate(model_names):
    for subj in range(n_subj):
        plot_data.append({'Model': modelname, 
                         'Condition': 'Target Objects', 
                         'Correlation': corrs_targets[subj, i]})
        plot_data.append({'Model': modelname, 
                         'Condition': 'Passive Objects', 
                         'Correlation': corrs_objects[subj, i]})

plot_df = pd.DataFrame(plot_data)



import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_1samp, ttest_rel
from statsmodels.stats.multitest import multipletests

# ---------- Utility function for significance stars ----------
def get_stars(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return ''

# --------- INPUTS ---------
# corrs_targets and corrs_objects: shape (n_subjects, n_models)
# model_names: list of model names
# plot_df: pandas dataframe with columns ['Model', 'Condition', 'Correlation']

n_models = len(model_names)
box_width = 0.2

# --------- ONE-SAMPLE TESTS ----------
# Fisher z-transform
corrs_targets_z = np.arctanh(corrs_targets)
corrs_objects_z = np.arctanh(corrs_objects)

# One-sample t-test vs 0 (greater)
tvals_targets, pvals_targets = ttest_1samp(corrs_targets_z, 0, axis=0, nan_policy='omit', alternative='greater')
tvals_objects, pvals_objects = ttest_1samp(corrs_objects_z, 0, axis=0, nan_policy='omit', alternative='greater')

# FDR correction
reject_t, pvals_corr_t, _, _ = multipletests(pvals_targets, alpha=0.05, method='fdr_bh')
reject_o, pvals_corr_o, _, _ = multipletests(pvals_objects, alpha=0.05, method='fdr_bh')


# --------- PAIRED TESTS (only if at least one condition significant) ----------
paired_pvals = np.full(n_models, np.nan)  # placeholder for all models

# Compute raw paired-test p-values
for i in range(n_models):
    if reject_t[i] or reject_o[i]:
        tval_diff, pval_diff = ttest_rel(corrs_targets_z[:, i], corrs_objects_z[:, i], nan_policy='omit')
        paired_pvals[i] = pval_diff

# Indices of models to correct
test_indices = np.where(~np.isnan(paired_pvals))[0]

# FDR correction across all tested models
if len(test_indices) > 0:
    reject_corr, pvals_corr, _, _ = multipletests(paired_pvals[test_indices], alpha=0.05, method='fdr_bh')
    # Save results back into arrays aligned with all models
    paired_reject = np.full(n_models, False)
    paired_pvals_corr = np.full(n_models, np.nan)
    paired_reject[test_indices] = reject_corr
    paired_pvals_corr[test_indices] = pvals_corr
else:
    paired_reject = np.full(n_models, False)
    paired_pvals_corr = np.full(n_models, np.nan)


# --------- PLOTTING ----------
fig, ax = plt.subplots(figsize=(6, 6))


sns.boxplot(data=plot_df, x='Model', y='Correlation', hue='Condition', 
            ax=ax, palette={'Target Objects': "#ed0d0d", 'Passive Objects': '#ffb41e'}, width=0.6, # colors were skyblue and salmon in previous version
            showfliers=False,
            gap=0.2) # hide outliers for clarity

# One-sample significance stars
for i, model_name in enumerate(model_names):
    # Targets
    if reject_t[i]:
        star = get_stars(pvals_corr_t[i])
        model_targets = plot_df[(plot_df['Model'] == model_name) & (plot_df['Condition'] == 'Target Objects')]['Correlation']
        y_pos = model_targets.max() + 0.02
        ax.text(i - box_width/2, y_pos, star, ha='center', va='bottom', color='#ed0d0d', fontsize=16)
    # Objects
    if reject_o[i]:
        star = get_stars(pvals_corr_o[i])
        model_objects = plot_df[(plot_df['Model'] == model_name) & (plot_df['Condition'] == 'Passive Objects')]['Correlation']
        y_pos = model_objects.max() + 0.02
        ax.text(i + box_width/2, y_pos, star, ha='center', va='bottom', color='#ffb41e', fontsize=16)

# Paired significance stars/brackets
for i, model_name in enumerate(model_names):
    if paired_reject[i]:
        # bracket line
        model_targets = plot_df[(plot_df['Model'] == model_name) & (plot_df['Condition'] == 'Target Objects')]['Correlation']
        model_objects = plot_df[(plot_df['Model'] == model_name) & (plot_df['Condition'] == 'Passive Objects')]['Correlation']
        y = max(model_targets.max(), model_objects.max()) + 0.06
        ax.plot([i - box_width/2, i + box_width/2], [y, y], color='black', lw=1.2)
        star = get_stars(paired_pvals_corr[i])
        ax.text(i, y + 0.01, star, ha='center', va='bottom', color='black', fontsize=14)

# Axes and formatting
ax.set_xticklabels([clean_modelnames_dictmap.get(name, name) for name in model_names]) # Rename x-axis tick labels
ax.set_ylabel(r"$\mathbf{Spearman's\ \rho}$", fontsize=12)
ax.set_xlabel('')
ax.axhline(0, color='k', lw=1, linestyle='--', alpha=0.5)
legend = ax.legend(title='Condition', fontsize=11)
legend.get_title().set_fontweight('bold')
ax.set_ylim(-0.12, 0.32)
for spine in ax.spines.values():
    spine.set_linewidth(1)

# Add line break before last space in model names to improve label readability
def wrap_at_last_space(name):
    name = clean_modelnames_dictmap.get(name, name)
    idx = name.rfind(' ')
    if idx == -1:
        return name
    return name[:idx] + '\n' + name[idx+1:]

ax.set_xticklabels([wrap_at_last_space(name) for name in model_names], fontsize=12)


fig.savefig(f'TopVox_results_boxplot.png', dpi=300)
